In [68]:
import pandas as pd
import importlib
import funciones as f #Tener funciones.py en mismo directorio. Tiene las funciones usadas para procesar un df
importlib.reload(f)

data = pd.read_csv('competition_data.csv')
submission = pd.read_csv('submission.csv')
submission_aux = pd.read_csv('submission.csv')

### Descomentar la siguiente celda la primera vez que se corre el notebook

In [69]:
# uri_to_ms_data = f.get_songs_durations(data)
# uri_to_ms_submission = f.get_songs_durations(submission)
# diccionario = {**uri_to_ms_data, **uri_to_ms_submission}

In [70]:
# Ejemplo de uso de procesar_df
submission = f.procesar_df(submission, diccionario)

c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:113: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[i, new_col] = suma / len(obs) if len(obs) > 0 else 0


In [72]:
submission.columns

Index(['Unnamed: 0', 'ts', 'platform', 'master_metadata_track_name',
       'master_metadata_album_artist_name', 'master_metadata_album_album_name',
       'spotify_track_uri', 'shuffle', 'hour', 'day_of_week', 'month', 'year',
       'is_iphone', 'reason_appload', 'reason_backbtn', 'reason_clickrow',
       'reason_fwdbtn', 'reason_playbtn', 'reason_remote', 'reason_trackdone',
       'reason_trackerror', 'reason_unknown', 'fwdbtn_seguidos',
       'trackdone_seguidos', 'fwdbtn_spree', 'trackdone_spree',
       'fwdbtn_prop_30', 'duration_ms', 'track_prop', 'artist_prop',
       'album_prop', 'hour day_of_week'],
      dtype='object')

In [ ]:
# Ordenar data cronológicamente
data = f.sort_by_ts(data)

In [ ]:
# KFold
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import numpy as np
import xgboost as xgb

def temporal_stratified_kfold(data, n_splits=5):
    '''
    Requiere: data esta ordenada cronologicamente y tiene una columna 'ts' con la fecha.
    Devuelve: los indices de entrenamiento y validación para cada fold en un KFold estratificado temporalmente.
    '''
    data['ts'] = pd.to_datetime(data['ts'])
    data['year'] = data['ts'].dt.year - 2000
    folds = []
    for fold in range(n_splits):
        idxs_train = []
        idxs_val = []
        for year, df_year in data.groupby('year'):
            df_year = df_year.sort_values('ts')

            fold_size = len(df_year) // n_splits
            val_start = fold * fold_size
            val_end   = (fold + 1) * fold_size if fold < n_splits - 1 else len(df_year)

            # .iloc aquí selecciona posiciones locales,
            # pero .index te devuelve los labels globales
            val_idx   = df_year.iloc[val_start:val_end].index
            train_idx = df_year.drop(val_idx).index

            idxs_train.extend(train_idx)
            idxs_val.extend(val_idx)
        folds.append((idxs_train, idxs_val))
    return folds

In [ ]:
kf = temporal_stratified_kfold(data, n_splits=5)

auc_scores = []
columns_to_drop = ['Unnamed: 0', 'ts','TARGET', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'platform']

for fold, tupla in enumerate(kf):
    train_idx, val_idx = tupla
    print(f"Fold {fold + 1}")

    train_fold = data.loc[train_idx].copy().reset_index(drop=True)
    val_fold = data.loc[val_idx].copy().reset_index(drop=True)

    # Aplicar pipeline modular a cada fold
    train_fold = f.procesar_df(train_fold, diccionario)
    val_fold = f.procesar_df(val_fold, diccionario)
    print('len trainfold',len(train_fold.columns))
    print('len valfold',len(val_fold.columns))

    # Entrenar el modelo
    clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic',
                            seed = 42,
                            eval_metric = 'auc',
                            early_stopping_rounds = 100)
    clf_xgb.fit(train_fold.drop(columns=columns_to_drop),
                train_fold['TARGET'],
                eval_set=[(val_fold.drop(columns=columns_to_drop), val_fold['TARGET'])],
                verbose=False
                )

    # Evaluar sobre validación
    preds = clf_xgb.predict_proba(val_fold.drop(columns=columns_to_drop))[:, 1]
    auc = roc_auc_score(val_fold['TARGET'], preds)
    auc_scores.append(auc)
    print(f"AUC Fold {fold + 1}: {auc:.4f}")

# Resultado final
print(f"\nAUC promedio: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")

Fold 1
len trainfold 32
len valfold 32
AUC Fold 1: 0.8584
Fold 2
len trainfold 32
len valfold 32
AUC Fold 2: 0.9060
Fold 3
len trainfold 32
len valfold 32
AUC Fold 3: 0.8990
Fold 4
len trainfold 32
len valfold 32
AUC Fold 4: 0.8960
Fold 5
len trainfold 32
len valfold 32
AUC Fold 5: 0.9162

AUC promedio: 0.8951 ± 0.0196
